In [1]:
import csv
import json


INPUT_CSV = "biodiversity_funding_supercleaned.csv"       # Replace with your CSV file
OUTPUT_JSONL = "funding_finetune_data.jsonl" # The file we'll create


In [2]:
def create_qa_pairs(row):
    """
    Given one CSV row (as a dict), return multiple prompt-completion pairs.
    Each pair references columns from the row.
    """

    # Pull the columns (use .get() to avoid KeyErrors if a column is missing)
    funder = row.get("Funder", "")
    funding_amount = row.get("Funding Amount", "")
    application_deadline = row.get("Application Deadline", "")
    funding_focus = row.get("Funding Focus", "")
    eligibility = row.get("Eligibility Criteria", "")
    application_process = row.get("Application Process", "")
    past_recipients = row.get("Past Recipients (if available)", "")
    contact_info = row.get("Contact Information", "")
    geographic_focus = row.get("Geographic Focus", "")
    gotchas = row.get("Gotchas", "")
    additional_notes = row.get("Additional Notes", "")

    # We will create a few Q/A pairs per row:
    qa_list = []

    # Q/A #1: Basic overview
    prompt_1 = (
        f"I need a grant or funding that supports {funding_focus}. "
        f"Does {funder} provide this kind of funding?"
    )
    completion_1 = (
        f"{funder} offers {funding_amount}. Their application deadline is {application_deadline}. "
        f"Eligibility: {eligibility}. Application process: {application_process}."
    )
    qa_list.append({"prompt": prompt_1, "completion": completion_1})

    # Q/A #2: Gotchas or special conditions
    prompt_2 = (
        f"Are there any special conditions or gotchas when applying to {funder}?"
    )
    completion_2 = (
        f"For {funder}, gotchas include: {gotchas}. "
        f"Additional notes: {additional_notes}."
    )
    qa_list.append({"prompt": prompt_2, "completion": completion_2})

    # Q/A #3: Past recipients, if available
    if past_recipients.strip():
        prompt_3 = f"Any past recipients for {funder}?"
        completion_3 = (
            f"Past recipients include: {past_recipients}. For more info, contact {contact_info}."
        )
        qa_list.append({"prompt": prompt_3, "completion": completion_3})

    # Q/A #4: Geographic focus, if not empty
    if geographic_focus.strip():
        prompt_4 = f"Does {funder} support projects in {geographic_focus}?"
        completion_4 = (
            f"{funder} focuses on {geographic_focus}. "
            f"Feel free to reach out via {contact_info} for more details."
        )
        qa_list.append({"prompt": prompt_4, "completion": completion_4})

    return qa_list

In [3]:
def main():
    with open(INPUT_CSV, "r", encoding="utf-8") as infile, \
         open(OUTPUT_JSONL, "w", encoding="utf-8") as outfile:
        
        reader = csv.DictReader(infile)
        
        for row in reader:

            qa_pairs = create_qa_pairs(row)
     
            
            # Write each Q/A pair as one JSON record per line
            for qa in qa_pairs:
                record = {
                    "prompt": qa["prompt"],
                    "completion": qa["completion"]
                }
                # Convert record to JSON, write it out
                outfile.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Done! Created JSONL file: {OUTPUT_JSONL}")

if __name__ == "__main__":
    main()

Done! Created JSONL file: funding_finetune_data.jsonl
